Here are both use cases explained with clear diagrams and architecture flows for your interview.

---

## 🚗 Use Case 1: Car Accident Photo Validation with GenAI

**Scenario:** A user uploads a photo of a car accident. The system must validate:
- Is this actually a car accident photo?
- Is the damage consistent with the claim?
- Does it match policy data (vehicle model, color, etc.)?

### Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                         USER (Mobile/Web App)                        │
│                    Uploads accident photo + claim info               │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                      API GATEWAY (AWS)                               │
│                   POST /claim/upload-photo                           │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    S3 BUCKET (Photo Storage)                         │
│              stores: accident_photos/claim_123.jpg                   │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    LAMBDA (Orchestrator)                             │
│  1. Fetch photo from S3                                              │
│  2. Fetch claim data from DynamoDB (policy, vehicle, history)        │
│  3. Call Bedrock with multimodal prompt                              │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐
│  Bedrock         │ │  DynamoDB        │ │  SageMaker       │
│  (Claude 3       │ │  (Claim Data)    │ │  (Custom CV      │
│   Sonnet -       │ │  - Policy #      │ │   Model)         │
│   Vision)        │ │  - Vehicle model │ │  - Damage        │
│                  │ │  - Claim history │ │    detection     │
│  Analyzes:       │ │  - Prior claims  │ │  - Fraud         │
│  - Is it a car?  │ │                  │ │    patterns      │
│  - Accident?     │ │                  │ │                  │
│  - Damage type?  │ │                  │ │                  │
│  - Severity?     │ │                  │ │                  │
└────────┬─────────┘ └────────┬─────────┘ └────────┬─────────┘
         │                    │                    │
         └────────────────────┼────────────────────┘
                              ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    VALIDATION ENGINE (Lambda)                        │
│  Rules:                                                              │
│  ✅ Photo shows a car?                                               │
│  ✅ Damage matches claimed incident?                                 │
│  ✅ Vehicle matches policy (make/model/color)?                       │
│  ✅ No signs of fraud (stock photo, manipulation)?                   │
│  ✅ Claim history consistent?                                        │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐
│  AUTO-APPROVE    │ │  MANUAL REVIEW   │ │  AUTO-REJECT     │
│  (High confidence│ │  (Medium         │ │  (Fraud detected │
│   + clean data)  │ │   confidence)    │ │   or mismatch)   │
└──────────────────┘ └──────────────────┘ └──────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│              NOTIFICATION (SNS/SES) + AUDIT LOG (CloudWatch)         │
│                   User notified of claim status                      │
└─────────────────────────────────────────────────────────────────────┘
```

### How It Works (Step by Step)

| Step | Action | AWS Service |
| :--- | :--- | :--- |
| 1 | User uploads accident photo | API Gateway + S3 |
| 2 | Lambda triggers, fetches photo + claim data | Lambda + DynamoDB |
| 3 | Multimodal LLM analyzes photo | Bedrock (Claude Vision) |
| 4 | Custom CV model checks damage patterns | SageMaker |
| 5 | Validation engine cross-checks all signals | Lambda |
| 6 | Decision: approve / review / reject | Step Functions |
| 7 | User notified, audit logged | SNS + CloudWatch |

### Sample Bedrock Prompt

```
You are an insurance claim validator. Analyze this accident photo.

Claim Data:
- Policy: POL-12345
- Vehicle: 2020 Honda Civic, Blue
- Claimed incident: Rear-end collision
- Claim amount: $4,500

Photo Analysis Required:
1. Is this a car? (yes/no)
2. Is there visible damage? (yes/no)
3. What type of damage? (scratch/dent/crush/broken glass)
4. Does the damage match a rear-end collision? (yes/no)
5. Does the vehicle color match the policy? (yes/no)
6. Any signs of photo manipulation? (yes/no)
7. Confidence score (0-100)

Return JSON.
```

### Output Example

```json
{
  "is_car": true,
  "has_damage": true,
  "damage_type": "rear_bumper_crush",
  "matches_incident": true,
  "color_match": true,
  "manipulation_detected": false,
  "confidence": 87,
  "decision": "AUTO_APPROVE"
}
```

---

## 📁 Use Case 2: RAG on S3 Files (Risk Analysis Q&A)

**Scenario:** You have risk analysis documents, reports, and other files in S3. A user asks questions like *"What is the risk analysis for Q3 2024?"* and the system must find the right file and answer.

### Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                         USER QUESTION                                │
│              "What is the risk analysis for Q3 2024?"                │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│                    API GATEWAY + LAMBDA                              │
│                   Receives user question                             │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│              QUERY PROCESSING (Lambda)                               │
│  1. Rewrite query for better retrieval                               │
│  2. Generate embedding for the question                              │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│           VECTOR DATABASE (OpenSearch / Pinecone / pgvector)         │
│                                                                      │
│  Stored embeddings of all document chunks:                           │
│  ┌────────────────────────────────────────────────────────────┐     │
│  │ chunk_id | text                          | embedding       │     │
│  │ 001      | "Q3 2024 risk analysis..."    | [0.12, 0.45...] │     │
│  │ 002      | "Q2 2024 risk factors..."     | [0.33, 0.21...] │     │
│  │ 003      | "Annual risk report..."       | [0.78, 0.09...] │     │
│  └────────────────────────────────────────────────────────────┘     │
│                                                                      │
│  Similarity search → Top-K relevant chunks                           │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│              RERANKER (Bedrock / Cohere)                             │
│         Re-orders chunks by true relevance to question               │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│              CONTEXT ASSEMBLY (Lambda)                               │
│  Build prompt:                                                       │
│  ┌────────────────────────────────────────────────────────────┐     │
│  │ System: You are a risk analysis assistant.                 │     │
│  │ Context: [Top 3-5 relevant chunks from vector DB]          │     │
│  │ Question: What is the risk analysis for Q3 2024?           │     │
│  │ Answer:                                                    │     │
│  └────────────────────────────────────────────────────────────┘     │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│              BEDROCK (Claude / Titan)                                │
│         Generates answer grounded in retrieved context               │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│              RESPONSE + CITATIONS                                    │
│  "Based on Q3_2024_Risk_Analysis.pdf (page 4): ..."                │
└─────────────────────────────────────────────────────────────────────┘
```

### The Ingestion Pipeline (One-Time / Scheduled)

Before users can ask questions, documents must be indexed:

```
┌─────────────────────────────────────────────────────────────────────┐
│                    S3 BUCKET (Source Documents)                      │
│  ┌──────────────┐ ┌──────────────┐ ┌──────────────┐                 │
│  │ Q3_2024_     │ │ Q2_2024_     │ │ Annual_      │                 │
│  │ Risk.pdf     │ │ Risk.docx    │ │ Report.pdf   │                 │
│  └──────────────┘ └──────────────┘ └──────────────┘                 │
└──────────────────────────────┬──────────────────────────────────────┘
                               │ S3 Event Trigger
                               ▼
┌─────────────────────────────────────────────────────────────────────┐
│              LAMBDA (Document Processor)                             │
│  1. Extract text (PDF/DOCX/CSV)                                      │
│  2. Clean and normalize                                              │
│  3. Chunk into 500-1000 token pieces                                 │
│  4. Generate embeddings (Bedrock Titan Embeddings)                   │
│  5. Store in vector DB + metadata in DynamoDB                        │
└──────────────────────────────┬──────────────────────────────────────┘
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
┌──────────────────┐ ┌──────────────────┐ ┌──────────────────┐
│  Vector DB       │ │  DynamoDB        │ │  S3 (Original    │
│  (Embeddings)    │ │  (Metadata)      │ │   Files)         │
│                  │ │  - file name     │ │                  │
│                  │ │  - upload date   │ │                  │
│                  │ │  - doc type      │ │                  │
│                  │ │  - quarter/year  │ │                  │
└──────────────────┘ └──────────────────┘ └──────────────────┘
```

### How It Works (Step by Step)

| Step | Action | AWS Service |
| :--- | :--- | :--- |
| 1 | Documents uploaded to S3 | S3 |
| 2 | S3 event triggers Lambda | Lambda |
| 3 | Text extracted, chunked, embedded | Bedrock Titan Embeddings |
| 4 | Embeddings stored in vector DB | OpenSearch / pgvector |
| 5 | Metadata stored in DynamoDB | DynamoDB |
| 6 | User asks question | API Gateway |
| 7 | Question embedded, similarity search | Vector DB |
| 8 | Top chunks reranked | Bedrock Reranker |
| 9 | LLM generates answer with context | Bedrock (Claude) |
| 10 | Answer returned with citations | Lambda → User |

### Handling "Risk Analysis" Specifically

When the user asks *"What is the risk analysis for Q3 2024?"*:

1. **Query embedding** → vector search finds chunks semantically similar to "risk analysis Q3 2024"
2. **Metadata filter** → narrows to documents tagged `type=risk_analysis`, `quarter=Q3`, `year=2024`
3. **Hybrid search** → combines keyword match ("risk analysis") + semantic similarity
4. **Reranker** → ensures the most relevant chunks rise to the top
5. **LLM** → generates a grounded answer like:

> *"According to Q3_2024_Risk_Analysis.pdf (page 4), the key risks identified were: 1) Supply chain disruption (High), 2) Currency fluctuation (Medium), 3) Regulatory changes (Medium)..."*

### Key Components Summary

| Component | Purpose | AWS Service |
| :--- | :--- | :--- |
| **Document Store** | Raw files | S3 |
| **Text Extraction** | PDF/DOCX → text | Lambda + Textract |
| **Chunking** | Split into digestible pieces | Lambda |
| **Embeddings** | Convert text → vectors | Bedrock Titan |
| **Vector DB** | Store + search embeddings | OpenSearch / pgvector |
| **Metadata Store** | Filter by type/date | DynamoDB |
| **Reranker** | Improve precision | Bedrock Reranker |
| **LLM** | Generate answer | Bedrock (Claude) |
| **Orchestration** | Coordinate steps | Step Functions / Lambda |

---

## 🎯 Interview Talking Points

1. **Use Case 1** is a **multimodal validation** problem — combines vision (photo) + structured data (claim) + rules (validation) + GenAI (reasoning).
2. **Use Case 2** is a **RAG pipeline** — ingestion (S3 → chunks → embeddings → vector DB) + retrieval (query → search → rerank → generate).
3. **Both use Bedrock** as the LLM layer, but for different purposes: vision analysis vs. text generation.
4. **Both need orchestration** — Step Functions or Lambda to coordinate multiple services.
5. **Both need guardrails** — PII redaction, confidence thresholds, human-in-the-loop for edge cases.
6. **Cost control** — token budgeting, caching, chunk size optimization.

Would you like me to dive deeper into any specific part, like the chunking strategy, vector DB choice, or Bedrock prompt engineering for either use case?